# Code for P. Hughes to investigate RFI in DRAGONS spectra
### A. Ordog, July 2024

## Load python packages to use:

In [ ]:
import h5py                     # For opening the data files
import numpy as np              # For various operations on numbers and arrays
from astropy.time import Time   # For operations on time data
import matplotlib.pyplot as plt # For making plots (graphs)

## Define some functions that we will use:

In [ ]:
def read_data(filename):

    ## This function reads in the data from the scan and organizes it
    ## into a 'dictionary' where it can be easily accessed.
    
    # Define the (empty) dictionary:
    data = {}
    
    # Open the file:
    file = h5py.File(filename,'r')
    
    # Read in the frequency and add it to the data dictionary:
    data['freq'] = file['data']['beam_0']['band_SB0']['frequency'][:]/1e6
    
    # Read in the main part of the data set:
    dataset = file['data']['beam_0']['band_SB0']['scan_0']

    # Read in the positions and add them to the data dictionary:
    data['dec']   = dataset['metadata']['declination']
    data['ra']    = dataset['metadata']['right_ascension']
    data['el']    = dataset['metadata']['elevation']
    data['az']    = dataset['metadata']['azimuth']
    
    # Read in the times and add them to the data dictionary:
    data['t_utc'] = dataset['metadata']['utc']
    data['t_mjd'] = Time(data['t_utc'], format='isot',scale='utc').mjd
    
    # Read in the spectrometer power data and add them to the data dictionary:
    data['RR']   = dataset['data'][:,0,:]
    data['LL']   = dataset['data'][:,1,:]
    data['reRL'] = dataset['data'][:,2,:]
    data['imRL'] = dataset['data'][:,3,:]
    
    # These ones not needed for now:
    #noise = dataset['metadata']['noise_state'] 
    #trim_flag = dataset['metadata']['trim_scan_flag']
    
    return data


## Read in an example spectrum from each of the three survey phases

In [ ]:
# The names and locations of the three files:
file_p1 = '/arc/projects/GMIMS/lbn-tests/dva_survey_phase1_0345_Jy.h5'
#file_p2 = '/srv/data/dva/survey_azimuth_scans_Jy_corr/dva_survey_phase2_0503_Jy.h5'
#file_p3 = '/srv/data/dva/survey_azimuth_scans_Jy_corr/dva_survey_phase3_0390_Jy.h5'

# Feed the Phase 1 file to the function we defined for reading in the data:
data_p1 = read_data(file_p1)

## Take a look at how the data are structured:

Let's see what the dictionary we got from 'read_data' contains. Here are all the 'keys' or data elements that are defined in the dictionary:

In [ ]:
print('The keys: ', data_p1.keys())

Then we can print out the data stored under any of those keys. For example, the frequencies:

In [ ]:
print('Frequencies: ', data_p1['freq'])

or the times:

In [ ]:
print('Times (UTC): ', data_p1['t_utc'])


or one of the position arrays:

In [ ]:
print('Telescope azimuth: ', data_p1['az'])

Notice that Python writes '...' when the arrays are too long to display. So how long are these arrays of numbers?

In [ ]:
print('Frequencies: ',data_p1['freq'].shape)
print('Times: ',data_p1['t_utc'].shape)
print('Azimuth points: ',data_p1['az'].shape)

^ This means there are 8250 frequency channels, and 1803 times that the telescope recorded data, with 1803 corresponding positions.

What about the actual data that tells us how much signal we have at each time and each frequency? Let's take a look at one of those products, 'RR':

In [ ]:
print('Shape of RR: ',data_p1['RR'].shape)

## Now let's try making some plots!

First, if we want a signal versus frequency plot (a spectrum), we need to pick one of the 1803 times that the telescope recorded. All arrays in Python are *indexed* starting from 0.
-  index 0 is the first number in the array
-  index 1 is the second number
-  index 2 is the third number
-  etc

Let's pick a random index for now, say idx=200. With this, we can take a look at the 201st time that the telescope recorded data in this scan. Let's see what time that happened at:

In [ ]:
idx = 200

# Here we apply the index to the time array before printing out the result:
print(data_p1['t_utc'][idx])

^ This tells us the 201st data point was recorded at 7:47:26 UTC on July 7, 2022

When we plot something, we have to make sure the arrays we plot against each other have the same length or same shape. We want to plot telescope signal versus frequency. What shape do these have?

In [ ]:
print(data_p1['freq'].shape)
print(data_p1['RR'].shape)

^ This makes sense. The signal was recorded for each of the 1803 times, for each of the 8250 channels. We need to pick out just the one time index we are interested in:

In [ ]:
print(data_p1['freq'].shape)

# Apply the index to the RR signal array:
print(data_p1['RR'][idx,:].shape)

Now they have the same shape and we are ready to plot!

In [ ]:
# Define the figure and the axis 'ax' which we will use
fig,ax = plt.subplots(1,1,figsize=(12,6))

x = data_p1['freq'] # plot the frequencies along the x-axis
y = data_p1['RR'][idx,:] # plot the data along the y-axis

# Do the actual plotting:
ax.plot(x,y)

# Set the limits of your axes:
ax.set_xlim(350,1030) # pick the min and max frequencies to plot
ax.set_ylim(1000,7000) # pick the min and max signal levels to plot

It's a bit hard to tell what's going on in the RFI channels. Let's zoom in on one set of them:

In [ ]:
# Define the figure and the axis 'ax' which we will use
fig,ax = plt.subplots(1,1,figsize=(12,6))

x = data_p1['freq'] # plot the frequencies along the x-axis
y = data_p1['RR'][idx,:] # plot the data along the y-axis

# Do the actual plotting:
ax.plot(x,y)
ax.scatter(x,y,s=5,color='black') # s just means the size of the dots

# Set the limits of your axes:
ax.set_xlim(350,390) # pick the min and max frequencies to plot
ax.set_ylim(2000,7000) # pick the min and max signal levels to plot
ax.grid() # This puts a grid on the plot to make it easier to read